PEAS Component           	Description
P – Performance Measure	Deliver/pick up packages, avoid flying during storms, visit all zones, correctly identify package-free zones, complete route, and generate reroute alerts when required.

E – Environment	6 campus zones (0–5), each having a package status and weather status.

A – Actuators	Pick up package, advance to next zone, hold position, and log a reroute alert.

S – Sensors	Sensor that detects the current zone's package status and weather status.

Property	    Classification	Why?
Observable	    Partially Observable	The drone can only sense the package and weather in its current zone, not other zones.

Deterministic	Deterministic	Given the current state and action, the next state follows the specified rules without randomness.

Static	       Dynamic	   The environment can change while the drone is operating, particularly weather/package status.

Discrete	   Discrete	  There are a finite number of zones (0–5) and finite actions/statuses.

Agents	     Single-Agent	  Only the drone is making decisions; the campus environment is not another intelligent agent.

In [1]:
class DroneEnvironment:

    def __init__(self, packages, weather):
        self.packages = packages
        self.weather = weather
        self.total_zones = len(packages)

    def get_package_status(self, zone):
        return self.packages[zone]

    def get_weather_status(self, zone):
        return self.weather[zone]

    def pick_up_package(self, zone):
        if self.packages[zone] == "Package":
            self.packages[zone] = "Empty"
            print("Package picked up from Zone", zone)

    def change_weather(self, zone, new_weather):
        self.weather[zone] = new_weather

In [4]:
packages = [
    "Empty",
    "Package",
    "Empty",
    "Package",
    "Empty",
    "Package"
]

weather = [
    "Clear",
    "Clear",
    "Stormy",
    "Clear",
    "Clear",
    "Clear"
]

env = DroneEnvironment(packages, weather)

In [3]:
class SimpleReflexDroneAgent:

    def __init__(self):
        self.current_zone = 0

    def act(self, environment):

        package = environment.get_package_status(self.current_zone)
        weather = environment.get_weather_status(self.current_zone)

        print("Current Zone:", self.current_zone)
        print("Weather:", weather)
        print("Package:", package)

        # Rule 1
        if weather == "Stormy":
            print("Action: HOLD POSITION")

        # Rule 2
        elif package == "Package":
            print("Action: PICK UP PACKAGE")
            environment.pick_up_package(self.current_zone)

        # Rule 3
        else:
            print("Action: ADVANCE")

            if self.current_zone < environment.total_zones - 1:
                self.current_zone += 1

        print("-------------------------")

In [5]:
drone = SimpleReflexDroneAgent()

for step in range(1, 9):

    print("STEP", step)

    drone.act(env)

STEP 1
Current Zone: 0
Weather: Clear
Package: Empty
Action: ADVANCE
-------------------------
STEP 2
Current Zone: 1
Weather: Clear
Package: Package
Action: PICK UP PACKAGE
Package picked up from Zone 1
-------------------------
STEP 3
Current Zone: 1
Weather: Clear
Package: Empty
Action: ADVANCE
-------------------------
STEP 4
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 5
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 6
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 7
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 8
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------


In [6]:
class ModelBasedDroneAgent:

    def __init__(self, total_zones):

        self.current_zone = 0

        # Internal state
        self.confirmed_clear = [False] * total_zones

        self.consecutive_hold = 0

        self.route_complete = False

    def check_route_complete(self):

        if all(self.confirmed_clear):
            self.route_complete = True
            print("Route Complete")

    def act(self, environment):

        if self.route_complete:
            return

        package = environment.get_package_status(self.current_zone)
        weather = environment.get_weather_status(self.current_zone)

        print("Current Zone:", self.current_zone)
        print("Weather:", weather)
        print("Package:", package)

        # RULE 1
        if weather == "Stormy":

            self.consecutive_hold += 1

            print("Action: HOLD POSITION")

            # RULE 5
            if self.consecutive_hold > 3:
                print("Reroute Alert")

        # RULE 2
        elif package == "Package":

            self.consecutive_hold = 0

            print("Action: PICK UP PACKAGE")

            environment.pick_up_package(self.current_zone)

            # After picking it up, zone is package-free
            self.confirmed_clear[self.current_zone] = True

        # RULE 3
        else:

            self.consecutive_hold = 0

            print("Action: ADVANCE")

            # Current zone has been confirmed empty
            self.confirmed_clear[self.current_zone] = True

            if self.current_zone < environment.total_zones - 1:
                self.current_zone += 1

        # RULE 4
        self.check_route_complete()

        print("-------------------------")

In [7]:
packages = [
    "Empty",
    "Package",
    "Empty",
    "Package",
    "Empty",
    "Package"
]

weather = [
    "Clear",
    "Clear",
    "Stormy",
    "Clear",
    "Clear",
    "Clear"
]

env = DroneEnvironment(packages, weather)

drone = ModelBasedDroneAgent(6)

In [8]:
for step in range(1, 20):

    print("STEP", step)

    # Storm clears at step 8
    if step == 8:
        env.change_weather(2, "Clear")
        print("Weather at Zone 2 changed to Clear")

    drone.act(env)

    if drone.route_complete:
        break

STEP 1
Current Zone: 0
Weather: Clear
Package: Empty
Action: ADVANCE
-------------------------
STEP 2
Current Zone: 1
Weather: Clear
Package: Package
Action: PICK UP PACKAGE
Package picked up from Zone 1
-------------------------
STEP 3
Current Zone: 1
Weather: Clear
Package: Empty
Action: ADVANCE
-------------------------
STEP 4
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 5
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 6
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
-------------------------
STEP 7
Current Zone: 2
Weather: Stormy
Package: Empty
Action: HOLD POSITION
Reroute Alert
-------------------------
STEP 8
Weather at Zone 2 changed to Clear
Current Zone: 2
Weather: Clear
Package: Empty
Action: ADVANCE
-------------------------
STEP 9
Current Zone: 3
Weather: Clear
Package: Package
Action: PICK UP PACKAGE
Package picked up from Zone 3
--------------